# Fine-tune DeepFilterNet3 on Vietnamese speech (VIVOS) + UrbanSound8K noise

Fine-tunes the pretrained **DeepFilterNet3** (48 kHz fullband) on in-domain data:
clean speech from VIVOS (16 kHz, upsampled) mixed on-the-fly with UrbanSound8K noise
at random SNRs. Uses the official `deepfilternet` pip package's own model + loss
(`df.loss.Loss`), so nothing is reimplemented and the result stays loadable by the
app (`df.enhance.init_df`).

**Before you run:** upload your data to Google Drive as `denoise_data.zip` —
a zip of `ml/data` (needs `voice/vivos/train/waves/...` and `noise/audio/*.wav`), ~2.2 GB.

**Runtime:** T4, ~30–60 min for the defaults (8 epochs x 3000 random 3 s pairs).
Runtime > Change runtime type > **T4 GPU**.

**Output:** a model folder in Drive (`models/<name>/{config.ini, checkpoints/, metadata.json}`)
in the same delivery format as `ml/models/` — copy it back and evaluate with the local harness.

In [5]:
#@title Configuration — edit these, then Run all
DRIVE_DATA_ZIP = "/content/drive/MyDrive/UET/denoise_data.zip"  # zip of ml/data (voice/vivos + noise)
DRIVE_OUT      = "/content/drive/MyDrive/UET/dfn3_finetune"     # checkpoints + final model land here
MODEL_NAME     = "dfn3_ft_vivos_v1"

SEGMENT_SECONDS = 3.0
BATCH_SIZE      = 8        # T4 16 GB handles this; lower to 4 if OOM
STEPS_PER_EPOCH = 3000     # random pairs per epoch (dataset length)
EPOCHS          = 8
LR              = 1e-4     # fine-tune LR (pretrained was 1e-3)
SNRS            = [-5, 0, 5, 10, 20, 40]
P_NOISE_ONLY    = 0.05     # fraction of segments that are pure noise (clean = silence)
NUM_EVAL_PAIRS  = 12
SEED            = 42

import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
import sys; print("python", sys.version.split()[0])

GPU: Tesla T4 | torch 2.14.1+cu130
python 3.13.16


In [2]:
%%bash
set -e
# DeepFilterLib (the Rust extension behind deepfilternet) has prebuilt wheels only for
# Python <= 3.11, and pip can't build its sdist (it pins maturin 0.11). So: use a prebuilt
# wheel if one exists, otherwise build the GitHub v0.5.6 tag with a current maturin.
pip install -q soundfile loguru appdirs packaging requests sympy maturin

if pip install -q --only-binary=:all: "deepfilterlib==0.5.6" 2>/dev/null; then
  echo "installed prebuilt deepfilterlib wheel"
else
  echo "no prebuilt wheel for this Python -> building from source (~3-6 min, one-time per session)"
  if ! command -v cargo >/dev/null 2>&1 && [ ! -x "$HOME/.cargo/bin/cargo" ]; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal >/dev/null 2>&1
  fi
  export PATH="$HOME/.cargo/bin:$PATH"
  rm -rf /content/DeepFilterNet /content/dfl_wheels
  git -c advice.detachedHead=false clone -q --depth 1 --branch v0.5.6 https://github.com/Rikorose/DeepFilterNet.git /content/DeepFilterNet
  export PYO3_USE_ABI3_FORWARD_COMPATIBILITY=1   # lets pyo3 0.19 build on Python 3.12+
  maturin build --release -m /content/DeepFilterNet/pyDF/Cargo.toml -o /content/dfl_wheels \
      > /content/maturin.log 2>&1 || { tail -40 /content/maturin.log; exit 1; }
  pip install -q /content/dfl_wheels/*.whl
fi

# --no-deps: its metadata pins numpy<2 / packaging<24, which would fight Colab's preinstalled stack
pip install -q --no-deps deepfilternet==0.5.6

# df/io.py imports torchaudio.backend.common, which torchaudio >= 2.9 removed
python - <<'PY'
import importlib.util, pathlib
p = pathlib.Path(importlib.util.find_spec("df").submodule_search_locations[0]) / "io.py"
s = p.read_text()
old = "from torchaudio.backend.common import AudioMetaData"
new = ("try:\n    from torchaudio.backend.common import AudioMetaData\n"
       "except Exception:  # torchaudio >= 2.9\n    AudioMetaData = Any")
if old in s and new not in s:
    p.write_text(s.replace(old, new)); print("patched", p)
else:
    print("io.py already ok")
PY
echo "install done"

no prebuilt wheel for this Python -> building from source (~3-6 min, one-time per session)
patched /usr/local/lib/python3.13/dist-packages/df/io.py
install done


In [3]:
import df, libdf, soundfile  # the pip package 'deepfilternet' is imported as 'df'
from df.version import version
from df.enhance import init_df  # fails here if the torchaudio patch didn't apply
print("deepfilternet", version, "| libdf OK")

deepfilternet 0.5.6 | libdf OK


In [6]:
from google.colab import drive
drive.mount("/content/drive")

import glob, os
if not os.path.isfile(DRIVE_DATA_ZIP):
    raise FileNotFoundError(f"{DRIVE_DATA_ZIP} not found - upload your data zip to Drive first")
!unzip -q -o "$DRIVE_DATA_ZIP" -d /content/data

# find the corpus dirs regardless of zip nesting (ml/data/... or bare voice/...)
train_dirs = glob.glob("/content/data/**/vivos/train", recursive=True)
noise_dirs = glob.glob("/content/data/**/noise", recursive=True)
assert train_dirs and noise_dirs, f"voice/vivos/train or noise dir not found in the zip"
CLEAN_DIR, NOISE_DIR = train_dirs[0], noise_dirs[0]
print("clean:", CLEAN_DIR, "\nnoise:", NOISE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
clean: /content/data/data/voice/vivos/train 
noise: /content/data/data/noise


In [7]:
import math, shutil
from pathlib import Path
import numpy as np
import torch
import soundfile as sf
from scipy.signal import resample_poly

from df.enhance import init_df, df_features, enhance, get_model_basedir
from df.loss import Istft, Loss
from df.model import ModelParams
from df.utils import as_complex, get_norm_alpha, detach_hidden
from df.modules import get_device

SR = 48000
SEG = int(SEGMENT_SECONDS * SR)

# init_df loads the model's config.ini (incl. loss weights) and downloads the
# pretrained checkpoint on first use. Loading the config BEFORE building the
# loss matters: Loss() reads its factors from that config.
model, df_state, _ = init_df(model_base_dir="DeepFilterNet3")
device = get_device()
p = ModelParams()
istft = Istft(p.fft_size, p.hop_size, torch.as_tensor(df_state.fft_window().copy())).to(device)
loss_fn = Loss(df_state, istft).to(device)   # MultiResSpec + LocalSnr per config.ini
NB_DF = getattr(model, "nb_df", getattr(model, "df_bins", p.nb_df))
ERB_FB = df_state.erb_widths()
ALPHA = get_norm_alpha(False)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-12)
print("model on", device, "| fft", p.fft_size, "hop", p.hop_size, "nb_erb", p.nb_erb, "nb_df", NB_DF)

def batch_features(noisy_batch, clean_batch):
    """[B,1,T] CPU wavs -> model inputs + complex clean spec. Verified shapes:
    spec [B,1,T,481,2], erb [B,1,T,32], spec_feat [B,1,T,96,2], clean [B,1,T,481]."""
    specs, erbs, spec_fs, cleans = [], [], [], []
    for nw, cw in zip(noisy_batch, clean_batch):
        s, e, f = df_features(nw, df_state, NB_DF, device=None)
        specs.append(s); erbs.append(e); spec_fs.append(f)
        cleans.append(torch.as_tensor(df_state.analysis(cw.numpy())).squeeze(0))
    spec_b  = torch.cat(specs).to(device)
    erb_b   = torch.cat(erbs).to(device)
    specf_b = torch.cat(spec_fs).to(device)
    clean_c = torch.stack(cleans).unsqueeze(1).to(device)
    return spec_b, erb_b, specf_b, clean_c

2026-11-12 15:17:35 | INFO     | DF | Running on torch 2.14.1+cu130
2026-11-12 15:17:35 | INFO     | DF | Running on host b4ec4100a42f
2026-11-12 15:17:35 | INFO     | DF | Loading model settings of DeepFilterNet3
2026-11-12 15:17:35 | INFO     | DF | Using DeepFilterNet3 model at /root/.cache/DeepFilterNet/DeepFilterNet3
2026-11-12 15:17:35 | INFO     | DF | Initializing model `deepfilternet3`
2026-11-12 15:17:36 | INFO     | DF | Found checkpoint /root/.cache/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2026-11-12 15:17:36 | INFO     | DF | Running on device cuda:0
2026-11-12 15:17:36 | INFO     | DF | Model loaded
model on cuda:0 | fft 960 hop 480 nb_erb 32 nb_df 96


In [8]:
#@title Data: on-the-fly mixing (same rules as ml/training/dataset.py)
def load48(path):
    x, sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if sr != SR:
        g = math.gcd(SR, int(sr))
        x = resample_poly(x, SR // g, int(sr) // g).astype(np.float32)
    return np.nan_to_num(x).astype(np.float32)

def active_region(x, win_s=0.05, thresh=0.1):
    """Trim leading/trailing near-silence; UrbanSound8K clips are often one
    event inside seconds of silence."""
    w = int(win_s * SR)
    if x.size <= w:
        return x
    n_win = x.size // w
    env = np.sqrt((x[: n_win * w].reshape(n_win, w) ** 2).mean(axis=1))
    act = env > thresh * (env.max() + 1e-10)
    if not act.any():
        return x
    first = int(np.argmax(act))
    last = int(len(act) - np.argmax(act[::-1]) - 1)
    return x[first * w : (last + 1) * w]

def crop_or_pad(x, n, rng):
    if x.size >= n:
        s = int(rng.integers(0, x.size - n + 1))
        return x[s : s + n]
    return np.pad(x, (0, n - x.size)).astype(np.float32)

def mix_pair(clean_path, noise_path, rng):
    clean = load48(clean_path)
    clean = crop_or_pad(clean, SEG, rng)          # short utts get a noise-only tail (like SNR=-inf)
    noise = active_region(load48(noise_path))
    if noise.size < SEG:
        noise = np.tile(noise, int(np.ceil(SEG / noise.size)) + 1)
    noise = crop_or_pad(noise, SEG, rng)
    if rng.random() < P_NOISE_ONLY:
        clean = np.zeros(SEG, dtype=np.float32)
    else:
        clean = clean * 10 ** (rng.uniform(-6, 6) / 20)   # gain jitter, like DATALOADER_GAINS
        snr = float(SNRS[int(rng.integers(0, len(SNRS)))])
        c_rms = float(np.sqrt(np.mean(clean ** 2))) + 1e-10
        noise = noise * (c_rms * 10 ** (-snr / 20) / (float(np.sqrt(np.mean(noise ** 2))) + 1e-10))
    noisy = np.clip(clean + noise, -1, 1).astype(np.float32)
    return noisy, clean.astype(np.float32)

clean_files = sorted(glob.glob(CLEAN_DIR + "/**/*.wav", recursive=True))
noise_files = sorted(glob.glob(NOISE_DIR + "/**/*.wav", recursive=True))
assert clean_files and noise_files, f"no wavs found: {len(clean_files)} clean, {len(noise_files)} noise"
print(f"{len(clean_files)} clean utts, {len(noise_files)} noise clips")

class PairDataset(torch.utils.data.Dataset):
    """Infinite-feel dataset: every __getitem__ mixes a fresh random pair."""
    def __init__(self, n_items, epoch=0):
        self.n, self._epoch = n_items, epoch
    def set_epoch(self, e):
        self._epoch = e
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        rng = np.random.default_rng((SEED + 1) * 1_000_003 + self._epoch * 7919 + i)
        c = clean_files[int(rng.integers(0, len(clean_files)))]
        n = noise_files[int(rng.integers(0, len(noise_files)))]
        nw, cw = mix_pair(c, n, rng)
        return torch.from_numpy(nw)[None], torch.from_numpy(cw)[None]

train_ds = PairDataset(STEPS_PER_EPOCH)
loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=2,
                                     pin_memory=True, drop_last=True,
                                     multiprocessing_context="fork")

# fixed eval pairs from VIVOS *test* speakers (disjoint from train speakers)
test_files = sorted(glob.glob(CLEAN_DIR.replace("/train", "/test") + "/**/*.wav", recursive=True))
eval_files = test_files if test_files else clean_files
eval_rng = np.random.default_rng(SEED + 99)
eval_pairs = []
while len(eval_pairs) < NUM_EVAL_PAIRS:
    c = eval_files[int(eval_rng.integers(0, len(eval_files)))]
    n = noise_files[int(eval_rng.integers(0, len(noise_files)))]
    nw, cw = mix_pair(c, n, eval_rng)
    if cw.any():  # skip noise-only draws: SI-SDR against a silent reference is meaningless (~-100 dB)
        eval_pairs.append((nw, cw))
print(f"{len(eval_pairs)} fixed eval pairs ready")

11660 clean utts, 1000 noise clips
12 fixed eval pairs ready


In [9]:
#@title Train
def si_sdr(ref, est):
    ref = ref - ref.mean(); est = est - est.mean()
    alpha = np.dot(ref, est) / (np.dot(ref, ref) + 1e-10)
    proj, noise = alpha * ref, est - alpha * ref
    return 10 * np.log10((proj @ proj + 1e-10) / (noise @ noise + 1e-10))

def eval_si_sdr(m):
    m.eval()
    s_noisy, s_enh = [], []
    for nw, cw in eval_pairs:
        audio = torch.from_numpy(nw)[None]
        enh_wav = enhance(m, df_state, audio).numpy()[0]
        s_noisy.append(si_sdr(cw, nw))
        s_enh.append(si_sdr(cw, enh_wav))
    return float(np.mean(s_noisy)), float(np.mean(s_enh))

out_root = Path(DRIVE_OUT); cp_dir = out_root / "checkpoints"; cp_dir.mkdir(parents=True, exist_ok=True)
base_noisy, _ = eval_si_sdr(model)
print(f"epoch 0 (pretrained): si_sdr noisy={base_noisy:.2f} (noise floor reference)")
best = -1e9
for epoch in range(EPOCHS):
    train_ds.set_epoch(epoch)
    model.train()
    running = []
    for it, (nw, cw) in enumerate(loader):
        spec_b, erb_b, specf_b, clean_c = batch_features(nw, cw)
        noisy_c = as_complex(spec_b)
        snrs = torch.zeros(spec_b.shape[0], device=device)  # only used for loss summaries
        opt.zero_grad()
        enh, m, lsnr, _ = model(spec=spec_b.clone(), feat_erb=erb_b, feat_spec=specf_b)
        loss = loss_fn(clean_c, noisy_c, as_complex(enh), m, lsnr, snrs=snrs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        detach_hidden(model)
        running.append(loss.item())
        if it % 50 == 0:
            print(f"epoch {epoch} step {it}/{STEPS_PER_EPOCH // BATCH_SIZE} loss={np.mean(running[-50:]):.3f}")
    noisy_s, enh_s = eval_si_sdr(model)
    print(f"== epoch {epoch}: train_loss={np.mean(running):.3f} | eval si_sdr {noisy_s:.2f} -> {enh_s:.2f} dB")
    torch.save(model.state_dict(), cp_dir / f"model_{epoch + 1}.ckpt")
    if enh_s > best:  # keep best by eval SI-SDR
        best = enh_s
        torch.save(model.state_dict(), cp_dir / f"model_{epoch + 1}.ckpt.best")
        best_ckpt = cp_dir / f"model_{epoch + 1}.ckpt.best"
        print(f"   new best ({best:.2f} dB) -> {best_ckpt.name}")
print("done. best eval si_sdr:", round(best, 2), "dB")

epoch 0 (pretrained): si_sdr noisy=17.51 (noise floor reference)
epoch 0 step 0/375 loss=0.769
epoch 0 step 50/375 loss=1.101
epoch 0 step 100/375 loss=0.998
epoch 0 step 150/375 loss=1.019
epoch 0 step 200/375 loss=1.100
epoch 0 step 250/375 loss=0.951
epoch 0 step 300/375 loss=1.010
epoch 0 step 350/375 loss=0.991
== epoch 0: train_loss=1.023 | eval si_sdr 17.51 -> 21.79 dB
   new best (21.79 dB) -> model_1.ckpt.best
epoch 1 step 0/375 loss=0.896
epoch 1 step 50/375 loss=0.894
epoch 1 step 100/375 loss=0.969
epoch 1 step 150/375 loss=0.954
epoch 1 step 200/375 loss=0.904
epoch 1 step 250/375 loss=0.976
epoch 1 step 300/375 loss=0.901
epoch 1 step 350/375 loss=0.995
== epoch 1: train_loss=0.942 | eval si_sdr 17.51 -> 22.72 dB
   new best (22.72 dB) -> model_2.ckpt.best
epoch 2 step 0/375 loss=1.121
epoch 2 step 50/375 loss=0.919
epoch 2 step 100/375 loss=0.902
epoch 2 step 150/375 loss=1.010
epoch 2 step 200/375 loss=0.941
epoch 2 step 250/375 loss=0.879
epoch 2 step 300/375 loss=0.97

In [10]:
#@title Export to the ml/models/ delivery format (on Drive)
out = out_root / "models" / MODEL_NAME
(out / "checkpoints").mkdir(parents=True, exist_ok=True)
shutil.copy(get_model_basedir("DeepFilterNet3") + "/config.ini", out / "config.ini")
import json, datetime
meta = {
    "name": MODEL_NAME,
    "version": "0.1.0",
    "arch": "dfnet",
    "native_sr": 48000,
    "channels": 1,
    "config": {"base": "DeepFilterNet3", "loss": "df.loss.Loss (MultiResSpec + LocalSnr)"},
    "license": "internal",
    "created": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "training_run": f"colab finetune: {EPOCHS} epochs x {STEPS_PER_EPOCH} pairs, lr={LR}, "
                    f"data=VIVOS-train + UrbanSound8K, eval si_sdr {base_noisy:.2f}->{best:.2f} dB",
    "notes": f"best ckpt by eval SI-SDR: {best_ckpt.name}; eval pairs from VIVOS test speakers",
}
(out / "metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
shutil.copy(best_ckpt, out / "checkpoints" / best_ckpt.name)
print("exported ->", out)
print("\nNext steps (back on your machine):")
print(f"  1. download {out} from Drive and place it at ml/models/{MODEL_NAME}")
print(f"  2. add to ml/eval/adapters.yaml:")
print(f"       {MODEL_NAME}:")
print(f"         module: adapters.dfnet")
print(f"         args: {{ model: models/{MODEL_NAME} }}")
print(f"  3. python -m eval.runner --manifest eval/manifests/paired_v2.json --runs dfn3 {MODEL_NAME} --data-dir data/eval/paired_v2")

exported -> /content/drive/MyDrive/UET/dfn3_finetune/models/dfn3_ft_vivos_v1

Next steps (back on your machine):
  1. download /content/drive/MyDrive/UET/dfn3_finetune/models/dfn3_ft_vivos_v1 from Drive and place it at ml/models/dfn3_ft_vivos_v1
  2. add to ml/eval/adapters.yaml:
       dfn3_ft_vivos_v1:
         module: adapters.dfnet
         args: { model: models/dfn3_ft_vivos_v1 }
  3. python -m eval.runner --manifest eval/manifests/paired_v2.json --runs dfn3 dfn3_ft_vivos_v1 --data-dir data/eval/paired_v2


## Notes & known limits

- **Bandwidth hazard**: VIVOS is 16 kHz, so fine-tuning teaches DFN3 that 8–24 kHz is
  silence. On real fullband 48 kHz input the fine-tuned model may over-attenuate HF.
  Judge with the local eval + your ears; if HF damage shows up, mix in fullband data
  or treat the model as narrowband.
- **What the loss is**: `df.loss.Loss` with the pretrained config's weights —
  multi-resolution spectrogram loss (500/500) + local-SNR loss (1e-3). Same objective
  the model was pretrained with.
- **Determinism**: pair mixing is seeded per (epoch, index); reruns reproduce data but
  not CUDA numerics exactly.
- Colab free tier disconnects: checkpoints are written to Drive every epoch, so re-running
  the train cell resumes nothing automatically — but you can reload any epoch ckpt via
  `model.load_state_dict(torch.load(...))` before continuing.
- Full evaluation (SI-SDR / STOI / pesq proxy on the frozen `paired_v2` manifest) happens
  locally with the harness — see the printed next steps.
- **Install**: `deepfilterlib` has no prebuilt wheel for Python >= 3.12 and its sdist can't be
  built by pip, so the install cell builds it from the GitHub tag (needs Rust, installed automatically;
  ~3-6 min per fresh session) and patches one torchaudio import in `df/io.py`.